# Примеры аугментаций (Kaggle IRT)

YAML: `configs/augmentation_example.yaml`  
Там для **каждого** `.mat` из Kaggle лежат ROI (bbox дефектов) и список spatial-аугментаций.

Ноутбук показывает:
1. исходный кадр + маску + ROI;
2. несколько случайных аугментаций (картинка и маска крутятся вместе).

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from irt_data.cache import build_cache
from irt_data.config import DatasetConfig, AugConfig, AugSpec
from irt_data.io_backend import NpyMemmapBackend, MaskReader
from irt_data.crops import RoiCropper
from irt_data.transforms import TransformPipeline

ROOT = Path('.').resolve()
CFG_PATH = ROOT / 'configs' / 'augmentation_example.yaml'
cfg = DatasetConfig.from_yaml(CFG_PATH)

print('videos in yaml:', len(cfg.files_meta))
print('augs:')
for a in cfg.augs.spatial:
    print(' ', a.name, a.params)

videos in yaml: 38
augs:
  HorizontalFlip {'p': 0.5}
  VerticalFlip {'p': 0.5}
  RandomRotate90 {'p': 0.5}
  Affine {'scale': [0.85, 1.15], 'translate_percent': [-0.08, 0.08], 'rotate': [-25, 25], 'border_mode': 0, 'p': 0.8}
  ElasticTransform {'alpha': 40, 'sigma': 6, 'p': 0.3}


## Синхронность image + mask

Все ауги из YAML — **геометрические**. Albumentations применяет их и к `image`, и к `mask` в одном вызове (`apply_features` / `apply_temporal`).

- Flip / Rotate90 → IoU = 1.0 (пиксель в пиксель)
- Affine / Elastic → IoU ≈ 0.999 (картинка — bilinear, маска — nearest; расхождение < 1 px)
- На клипе все кадры получают **одну** геометрию через `images=[...]`

Фотометрию (Brightness, GaussNoise) в этот список не кладём: она должна менять только картинку, не маску.


In [2]:
from irt_data.transforms import TransformPipeline
from irt_data.config import AugConfig, AugSpec

def _iou(a, b):
    a, b = a > 0, b > 0
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union else 1.0

img = np.zeros((128, 160), np.float32); img[40:70, 55:95] = 1.0
msk = np.zeros((128, 160), np.uint8); msk[40:70, 55:95] = 3

print('IoU после каждой аугментации (p=1):')
for spec in cfg.augs.spatial:
    forced = AugSpec(spec.name, {**spec.params, 'p': 1.0})
    pipe = TransformPipeline(AugConfig(spatial=[forced]))
    scores = []
    for s in range(20):
        np.random.seed(s)
        oi, om = pipe.apply_features(img[..., None], msk.copy())
        scores.append(_iou(oi[..., 0] > 0.5, om > 0))
    print(f'  {spec.name:20s}  min IoU={min(scores):.4f}')


IoU после каждой аугментации (p=1):
  HorizontalFlip        min IoU=1.0000
  VerticalFlip          min IoU=1.0000
  RandomRotate90        min IoU=1.0000
  Affine                min IoU=0.9983
  ElasticTransform      min IoU=0.9991


## Окно интересной части видео

Используй уже существующий параметр `frame_range: [start, end)`:

```yaml
temporal:
  frame_range: [0, 200]   # глобально для всех файлов

files_meta:
  R_002:
    frame_range: [5, 80]  # только для этого файла (перебивает global)
    rois: [...]
```

Приоритет: **per-file** `files_meta.*.frame_range` → `temporal.frame_range` → всё видео.


## 1. Кэш кадров (нужны `.npy` для быстрого доступа)

Если кэша ещё нет — соберём несколько файлов для демо. Для всех 38:

```bash
python -m irt_data.cache --sources archive/data --out artifacts/cache
```

In [ ]:
demo_ids = ['R_002', 'R_003', 'R_004']
mat_paths = [ROOT / 'archive/data' / f'{vid}.mat' for vid in demo_ids]
build_cache(mat_paths, out_dir=ROOT / 'artifacts/cache', overwrite=False)

backend = NpyMemmapBackend(ROOT / 'artifacts/cache')
mask_reader = MaskReader({vid: ROOT / 'archive/labels/manual_mask' for vid in demo_ids})
for vid in demo_ids:
    mask_reader.register_alias(vid, vid)

print('cached:', [v for v in demo_ids if v in backend.list_ids()])

cache .mat -> .npy:   0%|          | 0/4 [00:00<?, ?it/s]

cached: ['R_002', 'R_010', 'Z_002', 'Z_012']


## 2. ROI из YAML на исходном кадре

In [ ]:
def draw_rois(ax, rois, colors=None):
    colors = colors or ['lime', 'cyan', 'orange', 'magenta', 'yellow', 'white']
    for i, roi in enumerate(rois):
        rect = patches.Rectangle(
            (roi.x, roi.y), roi.w, roi.h,
            linewidth=1.5, edgecolor=colors[i % len(colors)],
            facecolor='none',
        )
        ax.add_patch(rect)
        ax.text(roi.x, max(0, roi.y - 3), f'#{i}', color=colors[i % len(colors)], fontsize=8)


fig, axes = plt.subplots(len(demo_ids), 3, figsize=(10, 3.2 * len(demo_ids)))
if len(demo_ids) == 1:
    axes = np.expand_dims(axes, 0)

for row, vid in enumerate(demo_ids):
    meta = cfg.files_meta[vid]
    t = meta.peak_contrast or 11
    T, H, W = backend.shape(vid)
    t = min(t, T - 1)
    frame = backend.read_frames(vid, [t])[0]
    # normalize for display
    f = (frame - frame.min()) / (frame.max() - frame.min() + 1e-8)
    mask = mask_reader.read(vid)

    axes[row, 0].imshow(f, cmap='inferno')
    axes[row, 0].set_title(f'{vid} frame {t}')
    axes[row, 1].imshow(mask, cmap='tab10', vmin=0, vmax=255)
    axes[row, 1].set_title('mask')
    axes[row, 2].imshow(f, cmap='inferno')
    axes[row, 2].imshow(np.ma.masked_where(mask == 0, mask), cmap='autumn', alpha=0.45)
    draw_rois(axes[row, 2], meta.rois)
    axes[row, 2].set_title(f'ROI x{len(meta.rois)}')
    for a in axes[row]:
        a.axis('off')

plt.tight_layout()
plt.show()

## 3. Кроп вокруг ROI + аугментации

Берём один кадр, кропаем по случайному ROI из YAML, затем крутим через пайплайн из конфига.  
Маска всегда едет вместе с картинкой.

In [ ]:
# пайплайн ровно из YAML
tf = TransformPipeline.from_config(cfg.augs)
cropper = RoiCropper(cfg.crop)

def show_augs(vid: str, n_augs: int = 6, seed: int = 0):
    meta = cfg.files_meta[vid]
    t = min(meta.peak_contrast or 11, backend.num_frames(vid) - 1)
    frame = backend.read_frames(vid, [t])[0].astype(np.float32)
    frame = (frame - frame.min()) / (frame.max() - frame.min() + 1e-8)
    mask = mask_reader.read(vid).astype(np.uint8)

    rng = np.random.default_rng(seed)
    box = cropper.plan(*frame.shape[:2], rng, meta)
    img0 = cropper.apply_image(frame, box)
    msk0 = cropper.apply_mask(mask, box)

    fig, axes = plt.subplots(2, n_augs + 1, figsize=(2.2 * (n_augs + 1), 4.5))

    axes[0, 0].imshow(img0, cmap='inferno')
    axes[0, 0].set_title('crop')
    axes[1, 0].imshow(msk0, cmap='tab10', vmin=0, vmax=255)
    axes[1, 0].set_title('mask')
    axes[0, 0].axis('off'); axes[1, 0].axis('off')

    for i in range(n_augs):
        # каждая колонка — новый random seed аугментации
        aug_img, aug_msk = tf.apply_features(
            img0[..., None].astype(np.float32),  # (H,W,1)
            msk0,
        )
        aug_img = aug_img[..., 0]
        axes[0, i + 1].imshow(aug_img, cmap='inferno')
        axes[0, i + 1].set_title(f'aug {i+1}')
        axes[1, i + 1].imshow(aug_msk, cmap='tab10', vmin=0, vmax=255)
        axes[0, i + 1].axis('off'); axes[1, i + 1].axis('off')

    plt.suptitle(f'{vid}: ROI crop → augmentations (image + mask)')
    plt.tight_layout()
    plt.show()


show_augs('R_002', n_augs=6, seed=1)
show_augs('Z_012', n_augs=6, seed=2)

## 4. То же на клипе (несколько кадров сразу)

`apply_temporal`: одна геометрия на все кадры клипа.

In [ ]:
vid = 'R_010'
meta = cfg.files_meta[vid]
T_total = backend.num_frames(vid)
# несколько кадров вокруг пика
center = meta.peak_contrast or 11
idxs = np.clip(np.arange(center - 2, center + 6), 0, T_total - 1)
frames = backend.read_frames(vid, idxs).astype(np.float32)
# per-clip norm
frames = (frames - frames.min()) / (frames.max() - frames.min() + 1e-8)
mask = mask_reader.read(vid).astype(np.uint8)

rng = np.random.default_rng(7)
box = cropper.plan(frames.shape[1], frames.shape[2], rng, meta)
frames_c = cropper.apply_frames(frames, box)
mask_c = cropper.apply_mask(mask, box)

frames_a, mask_a = tf.apply_temporal(frames_c, mask_c)

n = min(6, len(frames_a))
fig, axes = plt.subplots(3, n, figsize=(2.1 * n, 6))
for i in range(n):
    axes[0, i].imshow(frames_c[i], cmap='inferno'); axes[0, i].set_title(f'in {idxs[i]}'); axes[0, i].axis('off')
    axes[1, i].imshow(frames_a[i], cmap='inferno'); axes[1, i].set_title(f'aug {i}'); axes[1, i].axis('off')
    axes[2, i].imshow(mask_a, cmap='tab10', vmin=0, vmax=255); axes[2, i].axis('off')
axes[0, 0].set_ylabel('crop')
axes[1, 0].set_ylabel('aug frames')
axes[2, 0].set_ylabel('aug mask')
plt.suptitle(f'{vid}: temporal clip — same transform for all frames')
plt.tight_layout()
plt.show()

## 5. Что в YAML

- `files_meta.<id>.rois` — bbox-ы дефектов (первый = объединение всех, дальше — отдельные компоненты)
- `augs.spatial` — albumentations по имени класса
- `crop.strategy: roi_random` — кроп вокруг случайного ROI

Чтобы добавить свою аугментацию — одна строка в YAML:

```yaml
- name: GridDistortion
  params: {p: 0.3}
```